# Runtime Policy Enforcement — Dev Log

## Objetivo

`@enforce(agent_id, action)` bloqueia DE VERDADE a execução de uma função se
`multi_agent_governance.authorize()` negar — diferente de `authorize()`
sozinho (consultivo), aqui é estruturalmente impossível rodar o corpo da
função sem autorização.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.runtime_policy_enforcement.enforcement import EnforcementError, enforce

@enforce(agent_id="auditor", action="audit_logs.verify_chain")
def verificar_cadeia():
    return "cadeia verificada com sucesso"

@enforce(agent_id="reviewer", action="policy_engine.evaluate")
def avaliar_politica():
    return "isto NUNCA deveria imprimir"

print("Chamada autorizada:", verificar_cadeia())
try:
    avaliar_politica()
except EnforcementError as e:
    print(f"Chamada bloqueada como esperado: {e}")

Chamada autorizada: cadeia verificada com sucesso
Chamada bloqueada como esperado: Ação 'policy_engine.evaluate' negada para o agente 'reviewer': Ação 'policy_engine.evaluate' NÃO está na lista de ações permitidas do agente 'Agente de Revisão Humana (interface)'.


## Testes e Handoff

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/runtime_policy_enforcement/tests -v
```

7/7 testes passando, incluindo prova de que o corpo da função NUNCA executa
quando não autorizado (efeito colateral observável ausente).

**Escopo honesto**: enforcement de aplicação, não kernel — código no mesmo
processo ainda poderia chamar a função original por outro caminho.